<a href="https://colab.research.google.com/github/carolshayle/Python/blob/main/Status_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
import io

In [2]:
# 1. Upload the Excel file
print("Please upload your 'Status_Report.xlsx' file:")
uploaded = files.upload()

# Read the Excel file into a pandas DataFrame
file_name = next(iter(uploaded))
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("\n--- Original DataFrame Head ---")
print(df.head())
print("\n--- DataFrame Info ---")
df.info()

# --- Data Cleaning/Preparation ---
# Ensure 'TOTAL_DONE' is numeric, coercing errors to NaN and then filling with 0
df['TOTAL_DONE'] = pd.to_numeric(df['TOTAL_DONE'], errors='coerce').fillna(0)

# Ensure 'STATUS' categories are consistent for better grouping and plotting
# Using .str.capitalize() to handle potential variations like "done", "Done", "DONE"
df['STATUS'] = df['STATUS'].astype(str).str.capitalize().replace({
    'Done': 'Done',
    'Ongoing': 'Ongoing',
    'Not started': 'Not Started' # Standardize "not started"
})

# Define colors for status categories for consistent plotting
status_colors = {
    'Done': 'green',
    'Ongoing': 'orange',
    'Not Started': 'red'
}


Please upload your 'Status_Report.xlsx' file:


Saving Status_Report.xlsx to Status_Report.xlsx

--- Original DataFrame Head ---
    District  Sub_distri STATUS    Assigned  TOTAL_DONE
0  Abdiaziiz  Dhagaxtuur   DONE  Abdisalaan       200.0
1  Abdiaziiz     Gaarisa   DONE  Abdisalaan       124.0
2  Abdiaziiz  Looyacadde   DONE  Abdisalaan       202.0
3  Abdiaziiz       Neero   DONE  Abdisalaan       352.0
4  Boondheer    Daljirka   DONE     Sabrina       251.0

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   District    93 non-null     object 
 1   Sub_distri  93 non-null     object 
 2   STATUS      93 non-null     object 
 3   Assigned    65 non-null     object 
 4   TOTAL_DONE  60 non-null     float64
dtypes: float64(1), object(4)
memory usage: 3.8+ KB


In [4]:
# 2. Bar graph for status per district with distinct colors
print("\n--- Bar Graph: Status Count per District ---")
status_district_counts = df.groupby(['District', 'STATUS']).size().unstack(fill_value=0)

fig_status_district = go.Figure()
# Ensure all expected statuses are added, even if a district doesn't have them
for status in ['Done', 'Ongoing', 'Not Started']:
    if status in status_district_counts.columns:
        fig_status_district.add_trace(go.Bar(
            x=status_district_counts.index,
            y=status_district_counts[status],
            name=status,
            marker_color=status_colors.get(status) # Use predefined colors
        ))

fig_status_district.update_layout(
    barmode='stack', # Stacks the bars for each district
    title_text='Status Count per District',
    xaxis_title="District",
    yaxis_title="Number of Sub-districts/Items",
    legend_title="Status",
    hovermode="x unified" # Improves hover experience
)
fig_status_district.show()


--- Bar Graph: Status Count per District ---


In [6]:
# 3. Table showing status per subdistrict
print("\n--- Table: Status per Subdistrict ---")
subdistrict_status_table = df[['District', 'Sub_distri', 'STATUS']].sort_values(by=['District', 'Sub_distri']).reset_index(drop=True)
print(subdistrict_status_table.to_string()) # .to_string() for full display in Colab output

# Display as an interactive Plotly table for better viewing within Colab
fig_table_sub = go.Figure(data=[go.Table(
    header=dict(values=list(subdistrict_status_table.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[subdistrict_status_table.District, subdistrict_status_table.Sub_distri, subdistrict_status_table.STATUS],
               fill_color='lavender',
               align='left'))
])
fig_table_sub.update_layout(title_text='Status per Subdistrict')
fig_table_sub.show()


--- Table: Status per Subdistrict ---
         District              Sub_distri       STATUS
0       Abdiaziiz              Dhagaxtuur         Done
1       Abdiaziiz                 Gaarisa         Done
2       Abdiaziiz              Looyacadde         Done
3       Abdiaziiz                   Neero         Done
4       Boondheer                Daljirka         Done
5       Boondheer             Nasiibuundo         Done
6       Boondheer                 Siinaay         Done
7       Boondheer       Yuusuf Al-kowneyn         Done
8     Daarusalaam              1da Luulyo  Not Started
9     Daarusalaam             Ceel mareer  Not Started
10    Daarusalaam                  Labiga  Not Started
11    Daarusalaam               Raliweyne  Not Started
12    Daarusalaam                 Tawakal  Not Started
13       Dayniile                Barwaaqo  Not Started
14       Dayniile             Ciisa Cabdi  Not Started
15       Dayniile              Daarusalam  Not Started
16       Dayniile         

In [7]:
# 4. Bar graph of number of digitized buildings per district
print("\n--- Bar Graph: Total Digitized Buildings per District ---")
buildings_per_district = df.groupby('District')['TOTAL_DONE'].sum().reset_index()
fig_buildings_district = px.bar(buildings_per_district,
                                x='District',
                                y='TOTAL_DONE',
                                title='Total Digitized Buildings per District',
                                labels={'TOTAL_DONE': 'Total Buildings Digitized', 'Districts': 'District'},
                                color='TOTAL_DONE', # Color intensity based on total done
                                color_continuous_scale=px.colors.sequential.Plasma, # A nice color scale
                                text='TOTAL_DONE') # Show values on top of bars
fig_buildings_district.update_layout(xaxis={'categoryorder':'total descending'}) # Order districts by total done
fig_buildings_district.show()



--- Bar Graph: Total Digitized Buildings per District ---


In [8]:
# 5. Pie chart of status in percentage
print("\n--- Pie Chart: Overall Status Distribution ---")
status_counts_percentage = df['STATUS'].value_counts(normalize=True).reset_index()
status_counts_percentage.columns = ['STATUS', 'Percentage']
status_counts_percentage['Percentage'] = status_counts_percentage['Percentage'] * 100 # Convert to actual percentage

fig_pie_status = px.pie(status_counts_percentage,
                        values='Percentage',
                        names='STATUS',
                        title='Overall Status Distribution',
                        color='STATUS', # Color slices by status category
                        color_discrete_map=status_colors, # Use predefined colors for consistency
                        hole=0.3) # Make it a donut chart
fig_pie_status.update_traces(textinfo='percent+label') # Show percentage and label on slices
fig_pie_status.show()


--- Pie Chart: Overall Status Distribution ---


In [9]:
# 6. Bar graph showing number of buildings per person assigned
print("\n--- Bar Graph: Total Digitized Buildings per Person Assigned ---")
buildings_per_person = df.groupby('Assigned')['TOTAL_DONE'].sum().reset_index()
fig_buildings_person = px.bar(buildings_per_person,
                              x='Assigned',
                              y='TOTAL_DONE',
                              title='Total Digitized Buildings per Person Assigned',
                              labels={'TOTAL_DONE': 'Total Buildings Digitized', 'Assigned': 'Person Assigned'},
                              color='TOTAL_DONE', # Color intensity by total done
                              color_continuous_scale=px.colors.sequential.Viridis, # Another nice color scale
                              text='TOTAL_DONE') # Show values on top of bars
fig_buildings_person.update_layout(xaxis={'categoryorder':'total descending'}) # Order by total done
fig_buildings_person.show()



--- Bar Graph: Total Digitized Buildings per Person Assigned ---


In [10]:
# 7. Table showing total number of buildings done per person assigned
print("\n--- Table: Total Digitized Buildings per Person Assigned ---")
buildings_per_person_table = df.groupby('Assigned')['TOTAL_DONE'].sum().reset_index()
buildings_per_person_table = buildings_per_person_table.sort_values(by='TOTAL_DONE', ascending=False).reset_index(drop=True)
print(buildings_per_person_table.to_string())

# Display as an interactive Plotly table
fig_table_person = go.Figure(data=[go.Table(
    header=dict(values=list(buildings_per_person_table.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[buildings_per_person_table.Assigned, buildings_per_person_table.TOTAL_DONE],
               fill_color='lavender',
               align='left'))
])
fig_table_person.update_layout(title_text='Total Digitized Buildings per Person Assigned')
fig_table_person.show()


--- Table: Total Digitized Buildings per Person Assigned ---
     Assigned  TOTAL_DONE
0  Abdisalaan     24377.0
1     Sabrina     15571.0
2     Sumaiya     15333.0
3  Abdirahman     10144.0
4      Ridwan      7866.0


In [11]:
# 8. Total number of buildings done (overall)
print("\n--- Total Number of Buildings Done (Overall) ---")
total_overall_buildings_done = df['TOTAL_DONE'].sum()
print(f"The grand total number of buildings digitized across all districts is: {int(total_overall_buildings_done)}")


--- Total Number of Buildings Done (Overall) ---
The grand total number of buildings digitized across all districts is: 73291
